# Proyecto Comportamiento de compras

## Carga de librerias y datasets

In [41]:
#Librerias

import pandas as pd


In [ ]:
#Datasets

df = pd.read_csv(r"C:\*****\*****\Projects\analysis-shopping-beheavior\data\raw\customer_shopping_behavior.csv")

In [43]:
#Copia de dataset

df_copy = df.copy()

## Exploración de datos

In [44]:
df.head()


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [45]:
print('Información general')
df.info()
print()
print('Lista de conteo de valores únicos')
print(df.nunique().sort_values(ascending=False))
print()
#Creación de Pivot Table nulos para 'users'
print('Lista de conteo de valores nulos')
tabla_orders_valores_nulos = pd.DataFrame({'Nulos': df.isna().sum().sort_values(ascending=False),
                                     'proporcion': df.isna().mean().sort_values(ascending=False)})
tabla_orders_valores_nulos['proporcion_%'] = (tabla_orders_valores_nulos['proporcion']*100).round(2).astype(str) + '%'
tabla_orders_valores_nulos = tabla_orders_valores_nulos.drop(columns='proporcion')

display(tabla_orders_valores_nulos)

Información general
<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 no

,Nulos,proporcion_%
Review Rating,37,0.95%
Customer ID,0,0.0%
Age,0,0.0%
Gender,0,0.0%
Category,0,0.0%
Item Purchased,0,0.0%
Location,0,0.0%
Size,0,0.0%
Color,0,0.0%
Purchase Amount (USD),0,0.0%


In [46]:
# Función para clasificar las columnas

def clasificar_columnas(df):
    """Clasifica columnas en numéricas, booleanas y categóricas."""

    columnas_ignorar = ['Customer ID']

    numericas, booleanas, categoricas = [], [], []

    for col in df.columns:

        #Condición para ignorar columnas
        if col in columnas_ignorar:
            continue

        valores_unicos = df[col].dropna().nunique()
        if valores_unicos <= 2:
            booleanas.append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            numericas.append(col)
        else:
            categoricas.append(col)

    return list(df.columns), numericas, booleanas, categoricas

# Llamada única — resultado reutilizado en todas las celdas siguientes

todas_columnas, numericas, booleanas, categoricas = clasificar_columnas(df)

# Tabla de clasificación con dict comprehension (un solo bucle)
mapeo_clasificacion = (
    {col: 'Numerica'   for col in numericas}
    | {col: 'Booleana'  for col in booleanas}
    | {col: 'Categórica' for col in categoricas}
)

tabla_variables = (
    pd.DataFrame(mapeo_clasificacion.items(), columns=['Columna', 'Tipo de variable'])
    .sort_values('Tipo de variable')
    .reset_index(drop=True)  # drop=True evita columna de índice redundante
)

print('Clasificación de las columnas')
display(tabla_variables)

Clasificación de las columnas


,Columna,Tipo de variable
0,Gender,Booleana
1,Subscription Status,Booleana
2,Discount Applied,Booleana
3,Promo Code Used,Booleana
4,Item Purchased,Categórica
5,Shipping Type,Categórica
6,Season,Categórica
7,Color,Categórica
8,Size,Categórica
9,Location,Categórica


### Resumen estadístico columnas numéricas

In [47]:
print( "Obtención de la mediana en Columnas numéricas\n\n", df[numericas].median(),"\n\n")
print("Resumen estadístico\n")
df[numericas].describe().round(0) # explorar columnas numéricas de df

Obtención de la mediana en Columnas numéricas

 Age                      44.0
Purchase Amount (USD)    60.0
Review Rating             3.8
Previous Purchases       25.0
dtype: float64 


Resumen estadístico



,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.0,3900.0,3863.0,3900.0
mean,44.0,60.0,4.0,25.0
std,15.0,24.0,1.0,14.0
min,18.0,20.0,2.0,1.0
25%,31.0,39.0,3.0,13.0
50%,44.0,60.0,4.0,25.0
75%,57.0,81.0,4.0,38.0
max,70.0,100.0,5.0,50.0


### Observaciones

La mediana como la media son valores iguales, en el caso de rating difiere un poco.

### Resumen estadítico columnas categóricas

In [48]:
print("Resumen estadístico\n")
df[categoricas].describe().round(0) # explorar columnas numéricas de df

Resumen estadístico



,Item Purchased,Category,Location,Size,Color,Season,Shipping Type,Payment Method,Frequency of Purchases
count,3900,3900,3900,3900,3900,3900,3900,3900,3900
unique,25,4,50,4,25,4,6,6,7
top,Blouse,Clothing,Montana,M,Olive,Spring,Free Shipping,PayPal,Every 3 Months
freq,171,1737,96,1755,177,999,675,677,584


## Tratamiento de datos

### Nulos

Para la columna rating 

In [49]:
# Columnas que presentaron valores nulos en el dataset de orders
df_columnas_con_nulos = ['Review Rating']

for columna in df_columnas_con_nulos:
    filas_nulas = df[df[columna].isna()]
    print(f"\n{'='*80}")
    print(f"Columna: '{columna}' - total de filas con valores nulos: {len(filas_nulas)}")
    print(f"{'='*80}")
    display(filas_nulas.head(20))  # Muestra solo las primeras 10 filas para evitar saturar la salida


Columna: 'Review Rating' - total de filas con valores nulos: 37


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
38,39,29,Male,Dress,Clothing,37,Florida,M,Red,Winter,NaN,Yes,2-Day Shipping,Yes,Yes,44,Venmo,Every 3 Months
50,51,49,Male,Blouse,Clothing,28,Maryland,M,Red,Spring,NaN,Yes,Store Pickup,Yes,Yes,39,Debit Card,Bi-Weekly
80,81,19,Male,Sandals,Footwear,72,New York,XL,Blue,Summer,NaN,Yes,Store Pickup,Yes,Yes,24,Credit Card,Every 3 Months
96,97,32,Male,Gloves,Accessories,73,Delaware,S,Gold,Spring,NaN,Yes,Express,Yes,Yes,43,Venmo,Every 3 Months
262,263,37,Male,Pants,Clothing,40,Hawaii,M,Teal,Summer,NaN,Yes,Next Day Air,Yes,Yes,6,Debit Card,Annually
330,331,46,Male,Sweater,Clothing,51,North Dakota,M,Beige,Winter,NaN,Yes,Express,Yes,Yes,43,Venmo,Bi-Weekly
356,357,56,Male,Pants,Clothing,80,Nevada,L,Black,Spring,NaN,Yes,Standard,Yes,Yes,27,Cash,Weekly
383,384,50,Male,Sunglasses,Accessories,77,West Virginia,S,Magenta,Fall,NaN,Yes,Standard,Yes,Yes,6,PayPal,Quarterly
419,420,53,Male,Shorts,Clothing,91,Utah,S,Teal,Winter,NaN,Yes,Free Shipping,Yes,Yes,14,Credit Card,Every 3 Months
442,443,29,Male,Sunglasses,Accessories,69,Michigan,M,Olive,Fall,NaN,Yes,Store Pickup,Yes,Yes,31,Credit Card,Quarterly


### Normalización nombre de columnas

In [50]:
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()
df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount', 'review_rating':'rating'})

df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season', 'rating',
       'subscription_status', 'shipping_type', 'discount_applied',
       'promo_code_used', 'previous_purchases', 'payment_method',
       'frequency_of_purchases'],
      dtype='str')

### Duplicados


In [51]:
print(f"Filas duplicadas (exactas): {df.duplicated().sum()}")

df = df.drop_duplicates()

Filas duplicadas (exactas): 0


### Normalización de frecuencia de dias

In [52]:
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90,
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping).astype('Int64') # Usar Int64 para permitir valores nulos

df[['frequency_of_purchases', 'purchase_frequency_days']].head(10)  # Convertir a str para evitar problemas de visualización

,frequency_of_purchases,purchase_frequency_days
0,Fortnightly,14
1,Fortnightly,14
2,Weekly,7
3,Weekly,7
4,Annually,365
5,Weekly,7
6,Quarterly,90
7,Weekly,7
8,Annually,365
9,Quarterly,90


## Segmentación de datos

### Edad

In [53]:
#Creación de la columna segmentada de edades de los clientes

labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)
df[['age', 'age_group']].head(10) 

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


### Exploración de descuentos y promos

In [54]:
df[['discount_applied', 'promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [55]:
#Verificación de uso de todas las promociones y descuentos en las compras

(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [56]:
df = df.drop('promo_code_used', axis=1)  # Ya que confirmé que se usaron todas las promociones y descuentos en las compras, no es necesario mantener la columna 'promo_code_used' en el dataset.

In [57]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season', 'rating',
       'subscription_status', 'shipping_type', 'discount_applied',
       'previous_purchases', 'payment_method', 'frequency_of_purchases',
       'purchase_frequency_days', 'age_group'],
      dtype='str')

## Conexión con PostgreSQL

In [ ]:
import sys
sys.path.append(r"C:\*****\*****\Projects\analysis-shopping-beheavior\src")

from load_df_to_postgres import load_dataframe_to_postgres

load_dataframe_to_postgres(
    df=df,
    table_name="customer_behavior",
    if_exists="replace",
    env_path=r"C:\*****\\Projects\analysis-shopping-beheavior\.env",
)

✓ Carga completada: 3900 filas en 'customer_behavior' (esperadas: 3900)
